# 32 build waveform pick events

32_build_event_catalog.py

STEP 32 of the Pinatubo FAIR pipeline.

Build the authoritative event catalog by integrating:
  - STEP 10 waveform index (waveform-defined event windows)
  - STEP 22 merged pick table (primary + secondary picks with provenance)
  - STEP 30 individual pick ↔ waveform map
    Used ONLY to bind authoritative analyst picks to waveform events.
    Step 32 never infers events from Step 30 — it only enriches waveform events.

Event classes:
  - WAV_ONLY    : waveform exists, no picks assigned
  - WAV+PICKS   : waveform exists, ≥1 pick assigned
  - PICKS_ONLY  : picks exist, no waveform matched/available

Event IDs:
  - waveform-defined events:  event_id = f"wav_{waveform_id}"
  - pick-only events:        event_id = f"pick_{pick_group_id}"  (pick_group_id == STEP22.event_id)

Hard guarantees:
  - No pick_id appears in more than one output event
  - No pick row is silently dropped (unassigned picks become PICKS_ONLY events)

Notes:
  - Authoritative pick times for individual picks are reconstructed from STEP 30 offsets
    and STEP 10 waveform start times (avoids PHA timestamp quirks).
  - Monthly picks are attached to waveform events only by time-window inclusion (± tolerance).
  - STEP 22 is assumed to have already suppressed monthly picks duplicating primary picks
    (seed_norm+phase within tolerance) at the global level.

Outputs:
  - Event CSV: one row per event
  - Pick map CSV: one row per assigned pick (individual + monthly), preserving provenance
  - QC CSV: one row per event, plus run-level summary

## Scope and execution

Waveform event classes, source priorities, monthly-block association and Step-30 timestamp reconstruction remain Pinatubo policy.

Review `DATA_TOP` and `SETTINGS`, then run the processing cell. Outputs may overwrite existing products according to the original step. The implementation is in the adjacent local module; no command-line script or subprocess is invoked.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display

# Find the adjacent local package whether Jupyter starts here or in the project root.
_candidates = [Path.cwd(), Path.cwd() / '10_conversion', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'pinatubo_steps').is_dir()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Open this notebook from the pipeline folder or project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from pinatubo_steps.config import pipeline_paths
from pinatubo_steps.step_32_build_waveform_pick_events import run

CONFIG_FILE = None  # Default: conversion_config.json; or set an explicit JSON path.
P = pipeline_paths(config_file=CONFIG_FILE)
DATA_TOP = Path(P['DATA_TOP'])
print('Configuration:', P['CONFIG_FILE'])
print('External outputs:', P['OUTPUT_ROOT'])

SETTINGS = dict(
    waveform_index=P['WAVEFORM_INDEX'],
    merged_picks=P['MERGED_PICK_INDEX'],
    individual_pick_waveform_map=P['INDIV_PICK_WAVEFORM_MAP'],
    out_event_csv=P['WAVEFORM_PICK_EVENT_INDEX'],
    out_pick_map_csv=P['PICK_MAP_CSV'],
    out_qc_csv=P['QC_CSV'],
    time_tolerance=0.5,
)
display(SETTINGS)


## Run the step

Imports and settings above do not run the analysis. This cell writes the configured products.


In [ ]:
result = run(**SETTINGS)


In [ ]:
# Preview returned tables and summary objects without reloading output files.
for name, value in result.items():
    print(name)
    display(value.head() if hasattr(value, 'head') else
            value[:5] if isinstance(value, list) else value)
